# Using Matrix Factorization for Counterfactual Species Pool Reconstruction

### 1. The Challenge: Quantifying the Unseen
Estimating **Dark Diversity**—the set of species absent from a site despite suitable conditions—requires reconstructing the **Species Pool** (the "Potential" distribution). Traditional methods often suffer from:

* **Environmental Confounding:** Failing to distinguish between shared environmental preferences and actual biotic interactions.
* **Subjectivity in Benchmarking:** Forcing researchers to manually define what a "natural" or "pristine" value looks like for complex variables such as canopy height or nutrient levels.
* **Data Sparsity:** Relying on pairwise co-occurrences that become unreliable in hyper-diverse, sparse communities.

### 2. The Solution: Residual-based Latent Counterfactuals
This implementation utilizes a **Joint Species Distribution Model (JSDM)** framework. Instead of manually defining every possible driver, we use **Matrix Factorization (MF)** to partition species distributions into two distinct mathematical components:

1.  **Abiotic Fixed Effects ($\mathbf{x}_i^\top \boldsymbol{\beta}_j$):** This represents the **Potential Distribution**. It captures the species' response to measured, fundamental environmental variables like Temperature, pH, and Elevation.
2.  **Latent Random Effects ($\mathbf{w}_i^\top \mathbf{z}_j$):** This acts as a "statistical vacuum cleaner," capturing all unmeasured deviations from the abiotic niche, such as land-use degradation, biotic competition, or dispersal limitation.

**The Integrated Model Equation:**
$$\text{logit}(p_{ij}) = \alpha_j + \mathbf{x}_i^\top \boldsymbol{\beta}_j + \mathbf{w}_i^\top \mathbf{z}_j$$

### 3. Reconstructing the Species Pool via Counterfactuals
The core of this method is the **Counterfactual Prediction**. To reconstruct the original "Potential" distribution (the Species Pool) without making subjective judgment calls on degradation, we:

* **Train with Latents:** Include the latent factors ($W, Z$) during training so they "soak up" the noise and stressors, preventing them from biasing the abiotic coefficients ($\beta$).
* **Predict without Latents:** Generate predictions using only the abiotic layer ($\alpha + X\beta$), effectively setting the unmeasured stressors to zero (the prior mean). 

This allows us to mathematically "undo" the impact of the unmeasured drivers of dark diversity to reveal the underlying environmental suitability.

### 4. Scalable Inference via SVI
To handle thousands of sites and species, we utilize **Stochastic Variational Inference (SVI)** in Pyro. SVI treats inference as an optimization problem, minimizing the difference between our approximation and the true posterior (ELBO optimization), which allows for efficient computation on high-dimensional ecological matrices even with limited computational resources.

In [14]:
import pandas as pd
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
import numpy as np

In [15]:
import os
from pathlib import Path

# Create output directory if it doesn't exist
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
print(f"Output directory: {output_dir.resolve()}")

Output directory: C:\Users\dyshe\OneDrive - Aarhus universitet\Documents\PhD\Chapter 1 - Biodiversity and restoration potential\Simulation_study\Matrix factorisation\output


In [16]:
# 1. Load and Prepare Data
survey_full_df = pd.read_csv('C:/Users/dyshe/OneDrive - Aarhus universitet/Documents/PhD/Chapter 1 - Biodiversity and restoration potential/Simulation_study/survey.csv', index_col=0)
survey_df = survey_full_df.drop(columns=['ID', 'x', 'y'])

env_df = pd.read_csv('C:/Users/dyshe/OneDrive - Aarhus universitet/Documents/PhD/Chapter 1 - Biodiversity and restoration potential/Simulation_study/env.csv', index_col=0)       # sites x env
# Drop ID and landuse columns - keep only abiotic variables
env_df = env_df.drop(columns=['ID', 'landuse'])

# Rescale the abiotic env data to have mean 0 and std 1
continuous_cols = env_df.columns.tolist()
env_df[continuous_cols] = (env_df[continuous_cols] - env_df[continuous_cols].mean()) / env_df[continuous_cols].std()

# Ensure all data is float type
env_df = env_df.astype(float)

# Convert to Tensors
# Y is the observation matrix (binary 0/1)
Y = torch.tensor(survey_df.values, dtype=torch.float32)
# X is the environmental matrix (Standardized abiotic variables only)
X = torch.tensor(env_df.values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-8)  # Add small epsilon to avoid division by zero

num_sites, num_species = Y.shape
num_env = X.shape[1]
num_factors = 5  # Number of latent factors to capture unmeasured drivers

print(f"Environmental variables: {env_df.columns.tolist()}")
print(f"Number of environmental covariates: {num_env}")
print(f"Number of latent factors: {num_factors}")


Environmental variables: ['temperature', 'pH', 'elevation']
Number of environmental covariates: 3
Number of latent factors: 5
Spatial coordinates shape: (225, 2)


In [17]:
# 2. Define the Model
def model(X, Y, num_factors):
    n_sites, n_species = Y.shape
    n_env = X.shape[1]

    # Priors for species-specific intercepts (baseline prevalence)
    alpha = pyro.sample("alpha", dist.Normal(0, 1).expand([n_species]).to_event(1))

    # Priors for Environmental coefficients (Beta)
    # Each species has its own response to each environmental variable
    beta = pyro.sample("beta", dist.Normal(0, 1).expand([n_env, n_species]).to_event(2))

    # Matrix Factorization Components
    # W: Site latent scores (The "environment" we didn't measure)
    W = pyro.sample("W", dist.Normal(0, 1).expand([n_sites, num_factors]).to_event(2))
    # Z: Species latent loadings (How species respond to unmeasured factors)
    Z = pyro.sample("Z", dist.Normal(0, 1).expand([n_species, num_factors]).to_event(2))

    # Linear Predictor: Intercept + Env Effects + Latent Factor Interaction
    # eta shape: (n_sites, n_species)
    eta = alpha + torch.matmul(X, beta) + torch.matmul(W, Z.t())

    # Likelihood: Bernoulli with Logit link
    with pyro.plate("sites", n_sites):
        pyro.sample("obs", dist.Bernoulli(logits=eta).to_event(1), obs=Y)

In [18]:
# 3. Setup Inference
pyro.clear_param_store()
guide = AutoDiagonalNormal(model)
optimizer = Adam({"lr": 0.01})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

In [19]:
# 4. Training Loop
num_iterations = 2500
for i in range(num_iterations):
    loss = svi.step(X, Y, num_factors)
    if i % 500 == 0:
        print(f"Iteration {i} - Loss: {loss:.2f}")

Iteration 0 - Loss: 20587.98
Iteration 500 - Loss: 6879.62
Iteration 1000 - Loss: 6638.01
Iteration 1500 - Loss: 6563.67
Iteration 2000 - Loss: 6539.01


In [20]:
# 5. Predict Species Distributions
# Get the "MAP" (Maximum A Posteriori) estimates from the guide
medians = guide.median()
alpha_hat = medians['alpha']
beta_hat = medians['beta']
W_hat = medians['W']
Z_hat = medians['Z']

# Calculate Predicted Probabilities
with torch.no_grad():
    # Full predictions: includes both environmental effects and latent factors
    logits_full = alpha_hat + torch.matmul(X, beta_hat) + torch.matmul(W_hat, Z_hat.t())
    probs_full = torch.sigmoid(logits_full)
    
    # Environmental-only predictions: excludes latent factors
    # This represents species' responses to measured environmental variables only
    logits_env_only = alpha_hat + torch.matmul(X, beta_hat)
    probs_env_only = torch.sigmoid(logits_env_only)

# Full predictions (observed diversity with all drivers)
predicted_probs_full_df = pd.DataFrame(probs_full.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_full_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_full.csv')

# Environmental-only predictions (potential diversity without unmeasured drivers)
predicted_probs_env_only_df = pd.DataFrame(probs_env_only.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_env_only_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_env_only.csv')

# Dark diversity proxy: difference between full and environmental-only
dark_diversity_proxy_df = predicted_probs_full_df - predicted_probs_env_only_df
dark_diversity_proxy_df.to_csv(output_dir / 'mat_fact_dark_diversity_proxy.csv')

print("Predictions generated:")
print(f"Full model (environment + latent): {predicted_probs_full_df.shape}")
print(f"  Mean probability: {predicted_probs_full_df.values.mean():.4f}")
print(f"\nEnvironment-only (no latent): {predicted_probs_env_only_df.shape}")
print(f"  Mean probability: {predicted_probs_env_only_df.values.mean():.4f}")
print(f"\nDark diversity proxy (full - env_only):")
print(f"  Mean difference: {dark_diversity_proxy_df.values.mean():.4f}")

Predictions generated:
Full model (environment + latent): (225, 100)
  Mean probability: 0.0937

Environment-only (no latent): (225, 100)
  Mean probability: 0.1814

Dark diversity proxy (full - env_only):
  Mean difference: -0.0877
